> ***Library***

In [2]:
import torch

## ***CUDA***

In [3]:
torch.cuda.is_available()

False

In [4]:
torch.cuda.device_count()

0

In [ ]:
#torch.cuda.get_device_name(0)
#torch.cuda.device(0)

## ***Tensors***

In [8]:
import numpy as np

arr = np.array([[1,2,3],[4,5,6]])
arr

array([[1, 2, 3],
       [4, 5, 6]])

In [9]:
tensor = torch.Tensor([[1,2,3],[4,5,6]])
tensor

tensor([[1., 2., 3.],
        [4., 5., 6.]])

In [10]:
arr * 5

array([[ 5, 10, 15],
       [20, 25, 30]])

In [11]:
tensor * 5

tensor([[ 5., 10., 15.],
        [20., 25., 30.]])

In [12]:
arr.sum()

np.int64(21)

In [13]:
tensor.sum()

tensor(21.)

In [16]:
tensor = torch.from_numpy(arr)
tensor

tensor([[1, 2, 3],
        [4, 5, 6]])

In [17]:
np.ones((2,4))

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.]])

In [18]:
torch.ones((2,4))

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [19]:
np.random.random((2,4))

array([[0.32673797, 0.96956651, 0.50212283, 0.09514641],
       [0.87755178, 0.26764598, 0.39925489, 0.91185813]])

In [20]:
torch.rand((2,4))

tensor([[0.5165, 0.9036, 0.6487, 0.2265],
        [0.3708, 0.7393, 0.7553, 0.0649]])

In [23]:
arr.shape

(2, 3)

In [24]:
arr.dtype

dtype('int64')

In [25]:
arr.device

'cpu'

In [26]:
tensor.shape

torch.Size([2, 3])

In [27]:
tensor.dtype

torch.int64

In [28]:
tensor.device

device(type='cpu')

In [31]:
#tensor.to("cuda")

In [32]:
tensor.numpy()

array([[1, 2, 3],
       [4, 5, 6]])

## ***Differentiation***

In [34]:
a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)

f = 3 * a**3 - b**2

$$f=3a^{3}-b^{2}$$

$$\frac{\partial f}{\partial a} = 9a^{2}$$

$$\frac{\partial f}{\partial b} = -2b$$

In [35]:
f

tensor([-12.,  65.], grad_fn=<SubBackward0>)

In [37]:
f.backward(gradient=torch.tensor([1,1]))

In [38]:
print(a.grad)

tensor([36., 81.])


In [39]:
9 * a**2

tensor([36., 81.], grad_fn=<MulBackward0>)

In [41]:
print(b.grad)

tensor([-12.,  -8.])


In [42]:
-2*b

tensor([-12.,  -8.], grad_fn=<MulBackward0>)

## ***Neural Networks***

In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [45]:
X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [48]:
print(X_train.shape)
print(y_train.shape)

(455, 30)
(455,)


In [50]:
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()

y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

In [51]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)

In [54]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [55]:
class BCNet(nn.Module):

    def __init__(self):
        super(BCNet, self).__init__()

        self.fc1 = nn.Linear(30, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.sigmoid(self.fc3(x))

        return x

In [56]:
model = BCNet()

In [57]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [58]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        preds = model(x_batch)
        loss = criterion(preds, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch {epoch+1}: loss was: {running_loss / len(train_loader)}')


Epoch 1: loss was: 0.6130770007769267
Epoch 2: loss was: 0.4760254661242167
Epoch 3: loss was: 0.31695199608802793
Epoch 4: loss was: 0.19482522209485373
Epoch 5: loss was: 0.1299607053399086
Epoch 6: loss was: 0.09454545763631662
Epoch 7: loss was: 0.07751166572173436
Epoch 8: loss was: 0.07249543964862823
Epoch 9: loss was: 0.06022925029198329
Epoch 10: loss was: 0.0557806296274066
Epoch 11: loss was: 0.05280162200021247
Epoch 12: loss was: 0.05612868424504995
Epoch 13: loss was: 0.047454699967056514
Epoch 14: loss was: 0.04511328365188092
Epoch 15: loss was: 0.04286969304084778
Epoch 16: loss was: 0.04111385637273391
Epoch 17: loss was: 0.0395758314213405
Epoch 18: loss was: 0.0374064402654767
Epoch 19: loss was: 0.03789247227832675
Epoch 20: loss was: 0.03384426158542434


In [60]:
with torch.no_grad():
    model.eval()

    preds = model(X_test_scaled_tensor)
    loss = criterion(preds, y_test_tensor).item()

    accuracy = ((preds >= 0.5) == y_test_tensor).float().mean().item()
print(accuracy)

0.9824561476707458
